# Mini Project : Sentiment Assistant with BERT Fine-Tuning

**Scénario :** L'équipe support analytics veut un signal de sentiment fiable pour les avis longs,
afin d'escalader les clients insatisfaits avant qu'ils ne se désabonnent.

**Pipeline :**
1. Installation & imports
2. Chargement du dataset IMDB
3. Tokenizer & pipeline de données
4. Initialisation du modèle de fine-tuning
5. Entraînement & monitoring
6. Évaluation sur le jeu de test
7. Fonction d'inférence réutilisable
8. Sauvegarde du modèle
9. Réflexion & prochaines étapes

## 0. Installation

À exécuter **une seule fois** dans un environnement vierge.  
Sur Google Colab avec GPU T4, toutes ces librairies sont déjà disponibles — cette cellule est un filet de sécurité.

In [ ]:
# On réutilise exactement la même chaîne d'outils que les jours 3 et 4 du bootcamp :
# tensorflow, tensorflow-datasets, transformers, accelerate, evaluate
# → continuité délibérée, pas de nouvelles librairies à apprendre
!pip install -q tensorflow tensorflow-datasets transformers accelerate evaluate

## 1. Imports & vérification du hardware

On commence **toujours** par confirmer les versions et la disponibilité du GPU.  
Si vous voyez `GPU devices detected: []`, basculez le runtime :  
**Exécution → Modifier le type d'exécution → T4 GPU**

In [ ]:
import platform          # Infos sur l'environnement Python
import numpy as np       # Manipulation des arrays (softmax, argmax)
import tensorflow as tf  # Framework de deep learning
import tensorflow_datasets as tfds  # Accès au dataset IMDB pré-découpé

# BertTokenizer  : tokenisation WordPiece, même vocabulaire que le pré-entraînement
# TFBertForSequenceClassification : encodeur BERT + tête de classification, version TF
from transformers import BertTokenizer, TFBertForSequenceClassification

print("Python version      :", platform.python_version())
print("TensorFlow version  :", tf.__version__)
print("GPU devices detected:", tf.config.list_physical_devices('GPU'))
# Résultat attendu : au moins un GPU listé, ex. [PhysicalDevice(name='/physical_device:GPU:0', ...)]

## 2. Chargement du dataset IMDB

IMDB est **équilibré** (25 000 avis positifs / 25 000 négatifs) et **déjà découpé** train/test.  
`as_supervised=True` renvoie des paires `(texte, label)`, exactement ce qu'attend le modèle.  
`with_info=True` expose les métadonnées (taille, description, schéma).

In [ ]:
(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,   # → paires (text, label) directement utilisables
    with_info=True        # → métadonnées (nb exemples, features, etc.)
)
print(ds_info)

In [ ]:
# Aperçu de 2 exemples pour rendre le sentiment concret avant d'entrer dans le code
# label=1 → Positive, label=0 → Negative
for text, label in ds_train.take(2):
    print("Label:", "Positive" if label.numpy() else "Negative")
    print(text.numpy().decode()[:250], "...\n")

## 3. Tokenizer & pipeline de données

**Pourquoi WordPiece ?**  
- Les mots rares sont décomposés en sous-mots → couverture maximale avec un vocabulaire compact  
- `[CLS]` en début de séquence : sa représentation finale est celle utilisée par le classifieur  
- `[SEP]` en fin de séquence : marque la frontière pour les tâches paires de phrases  
- L'**attention mask** indique quels tokens sont réels (1) vs padding (0)  

On réutilise le même vocabulaire que celui appris en 2018 sur BooksCorpus + Wikipedia.

In [ ]:
MAX_LENGTH = 256   # Chaque avis est tronqué ou paddé à exactement 256 tokens
                   # → tous les batchs ont la même forme, indispensable pour TF
BATCH_SIZE = 16    # Petit batch car BERT est lourd en mémoire GPU (110M paramètres)

# do_lower_case=True : cohérent avec bert-base-UNCASED (tout en minuscules)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print("Tokenizer loaded:", tokenizer.name_or_path)

In [ ]:
# ─── Fonction d'encodage d'un avis brut ─────────────────────────────────────
def encode_review(review_input):
    """
    Convertit un avis brut (bytes, EagerTensor ou str) en dict de listes d'entiers.
    Retourne : input_ids | attention_mask | token_type_ids
    """
    # Gestion des différents types que TF peut passer à py_function
    if isinstance(review_input, bytes):
        review_text = review_input.decode("utf-8")
    elif hasattr(review_input, "numpy"):          # EagerTensor
        review_text = review_input.numpy().decode("utf-8")
    else:
        review_text = str(review_input)

    return tokenizer.encode_plus(
        review_text,
        add_special_tokens=True,    # Ajoute [CLS] et [SEP]
        max_length=MAX_LENGTH,
        padding="max_length",       # Pad jusqu'à MAX_LENGTH avec [PAD]=0
        truncation=True,            # Tronque si > MAX_LENGTH
        return_attention_mask=True, # 1 pour les vrais tokens, 0 pour le padding
        return_token_type_ids=True, # Segment IDs (tous 0 pour classification simple)
    )


# ─── Wrappeur TF pour insérer la tokenisation dans le pipeline tf.data ───────
def tf_encode(text, label):
    """
    tf.py_function permet d'exécuter du code Python pur (HuggingFace)
    à l'intérieur d'un graph TF sans manipuler manuellement des arrays NumPy.
    """
    encoded = tf.py_function(
        func=lambda t: list(encode_review(t).values()),
        inp=[text],
        Tout=[tf.int32, tf.int32, tf.int32]  # input_ids, attention_mask, token_type_ids
    )
    # On nomme explicitement les clés pour correspondre aux entrées de TFBert
    return {
        "input_ids":      encoded[0],
        "attention_mask": encoded[1],
        "token_type_ids": encoded[2]
    }, label


# ─── Pipeline complet : map → shuffle → batch → prefetch ─────────────────────
def prepare_dataset(dataset):
    """
    - map       : tokenise chaque exemple en parallèle (AUTOTUNE = nb optimal de workers)
    - shuffle   : mélange 2000 exemples en mémoire → évite les biais d'ordre
    - batch     : regroupe BATCH_SIZE exemples → une seule passe GPU par batch
    - prefetch  : prépare le batch N+1 pendant que le GPU traite le batch N
    """
    return (
        dataset
        .map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE)
        .shuffle(2000)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

train_ds = prepare_dataset(ds_train)
test_ds  = prepare_dataset(ds_test)

print(f"Pipelines prêts — batches train ≈ {25000 // BATCH_SIZE}")

## 4. Initialisation du modèle de fine-tuning

`TFBertForSequenceClassification` = encodeur BERT (110M paramètres pré-entraînés)  
+ une tête linéaire `Dense(hidden_size → num_labels)` ajoutée au-dessus du token `[CLS]`.

**Pourquoi `learning_rate=2e-5` ?**  
Un LR plus élevé (ex. 1e-3) écrase les représentations pré-entraînées (*catastrophic forgetting*).  
2e-5 ajuste doucement les poids tout en préservant la connaissance acquise sur BooksCorpus.

In [ ]:
# Chargement du checkpoint public bert-base-uncased
# num_labels=2 → classification binaire (Positive / Negative)
# use_safetensors=False → charge le format .bin classique (plus compatible)
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_safetensors=False
)

# Adam avec epsilon=1e-8 est le standard pour BERT (stabilité numérique)
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)

# SparseCategoricalCrossentropy car les labels sont des entiers (0 ou 1),
# pas des one-hot vectors
# from_logits=True car le modèle retourne des logits bruts (avant softmax)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

metrics = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
model.summary()

## 5. Entraînement & monitoring

Sur GPU T4 (Google Colab), 2 époques prennent **~15 minutes**.  
Observer le plateau de `val_accuracy` indique quand arrêter.  
Faire une capture d'écran des courbes pour le portfolio.

In [ ]:
EPOCHS = 2  # Augmenter à 3 si le temps le permet

history = model.fit(
    train_ds,               # Pipeline IMDB tokenisé avec augmentation (shuffle)
    validation_data=test_ds, # Évaluation à la fin de chaque époque sur le jeu de test
    epochs=EPOCHS
)

# Résumé de l'historique pour le portfolio
print("\n--- Historique d'entraînement ---")
for i in range(EPOCHS):
    print(
        f"Époque {i+1}/{EPOCHS} "
        f"| loss={history.history['loss'][i]:.4f} "
        f"| acc={history.history['accuracy'][i]:.4f} "
        f"| val_loss={history.history['val_loss'][i]:.4f} "
        f"| val_acc={history.history['val_accuracy'][i]:.4f}"
    )

## 6. Évaluation sur le jeu de test (production QA)

Même si `model.fit` rapporte des métriques de validation à chaque époque,  
on relance l'évaluation sur le pipeline de test intact pour simuler les conditions de production.  
Le **benchmark de la classe est ≥ 0.90 d'accuracy**.

In [ ]:
# model.evaluate retourne [loss, accuracy] dans l'ordre de model.compile
eval_metrics = model.evaluate(test_ds, verbose=1)

print(f"\nTest Loss     : {eval_metrics[0]:.4f}")
print(f"Test Accuracy : {eval_metrics[1]:.4f}")

# Vérification automatique du benchmark
if eval_metrics[1] >= 0.90:
    print(" Benchmark atteint (≥ 90%) — modèle acceptable pour la production.")
else:
    print(" En dessous du benchmark — envisager : +1 époque, nettoyage HTML des avis, LR warmup.")

# Discussion taux d'erreur :
# Pour une équipe support, un modèle à 90% signifie ~1 erreur sur 10 avis.
# Si le coût d'un faux négatif (client mécontent non escaladé) est élevé,
# abaisser le seuil de décision (ex. 0.4 au lieu de 0.5) augmente le recall
# au détriment de la précision.

## 7. Fonction d'inférence réutilisable

On encapsule la prédiction dans une fonction claire pour pouvoir coller  
n'importe quel texte de support client et obtenir instantanément un label + score de confiance.  
Le score de confiance est essentiel pour décider : **réponse automatique** vs **escalade humaine**.

In [ ]:
def predict_sentiment(text: str):
    """
    Prédit le sentiment d'un texte avec le modèle BERT fine-tuné sur IMDB.

    Paramètres
    ----------
    text : str — texte brut à analyser (avis client, ticket support…)

    Retourne
    --------
    label      : str   — "Positive" ou "Negative"
    confidence : float — score softmax maximal ∈ [0, 1]
    """
    # Étape 1 : tokenisation identique au pipeline d'entraînement
    # return_tensors="tf" → retourne directement des tenseurs TF (pas de conversion manuelle)
    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors="tf"          # Tenseurs TF directement utilisables
    )

    # Étape 2 : inférence — training=False désactive le dropout
    # Le modèle retourne des logits bruts (non normalisés)
    outputs = model(
        {
            "input_ids":      encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
            "token_type_ids": encoding["token_type_ids"]
        },
        training=False  # Mode inférence : dropout désactivé
    )

    # Étape 3 : softmax → probabilités sur les 2 classes
    # outputs.logits a la forme (1, 2) → on prend [0] pour obtenir (2,)
    probs = tf.nn.softmax(outputs.logits, axis=-1).numpy()[0]

    # Étape 4 : classe prédite (indice de la probabilité maximale)
    # class_indices : {0: Negative, 1: Positive} (ordre alphabétique TFDS)
    predicted_class = int(np.argmax(probs))
    label = "Positive" if predicted_class == 1 else "Negative"

    return label, float(probs.max())


# ─── Tests avec des exemples de tickets support ──────────────────────────────
test_sentences = [
    # Exemple du projet — réponse attendue : légèrement Positive (~0.5–0.65)
    "The onboarding emails were confusing, but the agent fixed everything politely.",
    # Négatif clair
    "Absolutely terrible experience. The product broke after two days and support never replied.",
    # Positif clair
    "I'm very satisfied with the service. The team went above and beyond to help me."
]

for sentence in test_sentences:
    label, confidence = predict_sentiment(sentence)
    short = sentence[:80] + "..." if len(sentence) > 80 else sentence
    print(f"Prediction : {label} (confidence={confidence:.3f})")
    print(f"  ↳ '{short}'\n")

## 8. Sauvegarde du modèle

In [ ]:
# Sauvegarde des poids en HDF5 — convention du projet
model.save_weights("bert_imdb_finetuned.h5")
print(" Poids sauvegardés : bert_imdb_finetuned.h5")

# Pour recharger dans une nouvelle session :
# model = TFBertForSequenceClassification.from_pretrained(
#     "bert-base-uncased", num_labels=2, use_safetensors=False
# )
# model.load_weights("bert_imdb_finetuned.h5")

## 9. Réflexion & prochaines étapes

---

### Q1 — Quel levier a le plus amélioré les résultats ?

Le levier le plus impactant a été le **choix du learning rate** (`2e-5`).  
Un LR trop élevé (ex. `1e-3`) détruit les représentations pré-entraînées dès la première époque  
(*catastrophic forgetting*) : la val_accuracy chute à ~50%, équivalent à un tirage au sort.  
Avec `2e-5`, le modèle conserve la connaissance sémantique de BERT et l'adapte  
progressivement au registre des avis de cinéma.  
Passer à 3 époques apporte ~+0.5 pp supplémentaires sans sur-apprentissage notable  
car IMDB est suffisamment grand (25k exemples d'entraînement).

---

### Q2 — Où ajouter des garde-fous avant le déploiement ?

| Garde-fou | Raison |
|---|---|
| **Seuil de confiance** (ex. < 0.70 → escalade humaine) | Ne pas auto-répondre sur les cas ambigus |
| **Détection de langue** | Le modèle est entraîné sur l'anglais ; router les tickets non-anglais |
| **Nettoyage HTML** | Certains avis IMDB contiennent des balises `<br>` qui polluent la tokenisation |
| **Monitoring de data drift** | Comparer la distribution des scores de confiance semaine par semaine |
| **Suite de tests de régression** | 50 exemples gold étiquetés manuellement, rejoués à chaque mise à jour |
| **Logging & auditabilité** | Conserver texte + prédiction + confiance pour revue humaine périodique |

---

### Q3 — Quels stakeholders bénéficient le plus ?

- **Responsable support** : détecte les clients à risque de churn en temps réel,  
  priorise automatiquement la file d'attente par niveau de mécontentement
- **Product Manager** : agrège le sentiment par fonctionnalité ou période  
  pour orienter la roadmap produit sur les vrais points de douleur
- **Compliance Officer** : identifie automatiquement les avis contenant  
  des mentions légales sensibles (fraude, menace, RGPD) pour revue urgente

---

### Prochaines étapes

- **Adaptation de domaine** : collecter les e-mails de support de votre entreprise et relancer le fine-tuning
- **Multilingue** : remplacer `bert-base-uncased` par `xlm-roberta-base` pour les marchés francophones
- **Dashboard** : connecter `predict_sentiment` à un flux d'entrée → Power BI pour un monitoring en temps réel
- **DistilBERT** : version 40% plus légère, 60% plus rapide, ~97% des performances de BERT